现在要回答的是两个问题：

1️⃣ 多重共线性是否严重？（VIF）
2️⃣ 省平均降水是否“吸收”了 Rx1day（极端降水）的解释力？

| VIF 值 | 含义    |
| ----- | ----- |
| < 5   | 基本无共线 |
| 5–10  | 中度共线  |
| > 10  | 严重共线  |
| > 20  | 非常严重  |


### **第一部分：计算 VIF（Variance Inflation Factor）**

In [1]:
# Cell 1：导入包
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from linearmodels.panel import PanelOLS


In [2]:
# Cell 2：读取 CSV
amr_path = r"C:\Users\lunch\Downloads\amr_rate.csv"
x_path   = r"C:\Users\lunch\Downloads\climate_social_eco.csv"

amr = pd.read_csv(amr_path, encoding="utf-8-sig")
x   = pd.read_csv(x_path, encoding="utf-8-sig")

print("AMR shape:", amr.shape)
print("X shape:", x.shape)


AMR shape: (313, 15)
X shape: (341, 31)


In [3]:
# Cell 3：统一主键列名
def normalize_key_cols(df):
    rename_dict = {}
    for c in df.columns:
        c_clean = c.strip().lower()
        if c_clean in ["province", "prov", "省份"]:
            rename_dict[c] = "Province"
        if c_clean in ["year", "yr", "年份"]:
            rename_dict[c] = "Year"
    df = df.rename(columns=rename_dict)
    return df

amr = normalize_key_cols(amr)
x   = normalize_key_cols(x)

# 清洗
amr["Province"] = amr["Province"].astype(str).str.strip()
x["Province"]   = x["Province"].astype(str).str.strip()

amr["Year"] = pd.to_numeric(amr["Year"], errors="coerce")
x["Year"]   = pd.to_numeric(x["Year"], errors="coerce")

# 限制年份
amr = amr[amr["Year"].between(2014, 2023)]
x   = x[x["Year"].between(2014, 2023)]

print("AMR years:", amr["Year"].unique())
print("X years:", x["Year"].unique())

AMR years: [2014 2015 2016 2017 2018 2019 2020 2021 2022 2023]
X years: [2014 2015 2016 2017 2018 2019 2020 2021 2022 2023]


In [5]:
# Cell 4：选择你要分析的9个变量
X_cols = [
    "省平均气温",
    "省平均降水",
    "R1xday",
    "PM2.5",
    "医疗水平",
    "GDP",
    "城市用水普及率",
    "生活垃圾无害化处理率",
    "抗菌药物使用强度"
]

missing = [c for c in X_cols if c not in x.columns]
print("缺失变量:", missing)

缺失变量: []


In [6]:
# Cell 5：合并面板数据
df = amr.merge(x[["Province", "Year"] + X_cols],
               on=["Province", "Year"],
               how="inner")

print("Merged shape:", df.shape)

Merged shape: (310, 24)


In [7]:
# Cell 6：构造综合 AMR
AMR_cols = [c for c in df.columns if c not in ["Province","Year"] + X_cols]

print("识别为AMR的列：", AMR_cols)

df["AMR_AGG"] = df[AMR_cols].mean(axis=1)

识别为AMR的列： ['MRCNS', 'VREFS', 'VREFM', 'PRSP', 'ERSP', '3GCRKP', 'MRSA', '3GCREC', 'CREC', 'QREC', 'CRPA', 'CRKP', 'CRAB']


In [8]:
# Cell 7：设置面板索引
df = df.set_index(["Province", "Year"]).sort_index()

In [9]:
# Cell 8：双向去均值函数
def twoway_demean(s):
    entity_mean = s.groupby(level=0).transform("mean")
    time_mean   = s.groupby(level=1).transform("mean")
    grand_mean  = s.mean()
    return s - entity_mean - time_mean + grand_mean

In [10]:
# Cell 9：计算 VIF（双向去均值后）
X_demeaned = pd.DataFrame(index=df.index)

for col in X_cols:
    X_demeaned[col] = twoway_demean(df[col])

X_demeaned = X_demeaned.dropna()

X_vif = sm.add_constant(X_demeaned)

vif_table = pd.DataFrame()
vif_table["Variable"] = X_vif.columns
vif_table["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

vif_table

,Variable,VIF
0,const,1.000722
1,省平均气温,1.125813
2,省平均降水,1.213897
3,R1xday,1.123010
4,PM2.5,1.107427
5,医疗水平,1.144303
6,GDP,1.284870
7,城市用水普及率,1.018397
8,生活垃圾无害化处理率,1.159866
9,抗菌药物使用强度,1.044716


省平均降水不能被其他变量线性解释

R1xday 也不能被降水高度解释

抗菌药物使用强度和 GDP/医疗水平 也不高度相关5

In [11]:
# Cell 10：检验“省平均降水是否解释R1xday”
# 方法1：相关系数
corr = X_demeaned["省平均降水"].corr(X_demeaned["R1xday"])
print("双向去均值后的相关系数:", corr)

双向去均值后的相关系数: 0.29285872242120975


In [12]:
# 方法2：固定效应检验
Y_test = twoway_demean(df["R1xday"])
X_test = twoway_demean(df["省平均降水"])

X_test = sm.add_constant(X_test)

mod = PanelOLS(Y_test, X_test,
               entity_effects=False,
               time_effects=False)

res = mod.fit()

print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                 R1xday   R-squared:                        0.0965
Estimator:                   PanelOLS   R-squared (Between):             -1.6336
No. Observations:                 310   R-squared (Within):               0.0965
Date:                Sun, Mar 01 2026   R-squared (Overall):              0.0965
Time:                        23:36:25   Log-likelihood                   -1293.2
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      32.882
Entities:                          31   P-value                           0.0000
Avg Obs:                      10.0000   Distribution:                   F(1,308)
Min Obs:                      10.0000                                           
Max Obs:                      10.0000   F-statistic (robust):             32.882
                            

In [15]:
# Cell 11：可选 —— 比较两个模型
# 模型A：只放R1xday
mod_A = PanelOLS(df["AMR_AGG"], df[["R1xday"]],
                 entity_effects=True,
                 time_effects=True)

res_A = mod_A.fit(cov_type="clustered", cluster_entity=True)

# 模型B：加降水
mod_B = PanelOLS(df["AMR_AGG"], df[["R1xday","省平均降水"]],
                 entity_effects=True,
                 time_effects=True)

res_B = mod_B.fit(cov_type="clustered", cluster_entity=True)

print("模型A结果")
print(res_A.summary)

print("模型B结果")
print(res_B.summary)

模型A结果
                          PanelOLS Estimation Summary                           
Dep. Variable:                AMR_AGG   R-squared:                     1.492e-05
Estimator:                   PanelOLS   R-squared (Between):             -0.0006
No. Observations:                 307   R-squared (Within):           -3.753e-05
Date:                Sun, Mar 01 2026   R-squared (Overall):             -0.0006
Time:                        23:45:02   Log-likelihood                   -425.65
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.0040
Entities:                          31   P-value                           0.9498
Avg Obs:                       9.9032   Distribution:                   F(1,266)
Min Obs:                       7.0000                                           
Max Obs:                      10.0000   F-statistic (robust):             0.0075
                      

d:\anaconda3\envs\iodide\lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
d:\anaconda3\envs\iodide\lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
